# Day 02 — Data Leakage: When Your Model Cheats

**Core question:** Can a model achieve an impressive test score and still fail in the real world?

We will use the Titanic dataset, continuing from Day 01. The dataset includes `alive`, which records the survival outcome, while our target is `survived`. Using `alive` as a predictor would reveal the answer rather than help predict it.

Today's workflow:
1. Understand the dataset statistically.
2. Investigate suspicious relationships.
3. Demonstrate target leakage.
4. Demonstrate preprocessing leakage.
5. Compare the wrong and correct workflows.
6. Record findings and limitations.

**Important:** statistical association is not enough to establish that a feature is usable. Ask whether the feature would genuinely be available at prediction time.

## 1. Imports and data loading

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = sns.load_dataset("titanic")
df = df.dropna(subset=["survived"]).reset_index(drop=True)

df.head()

## 2. Understand the dataset

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe(include="all").T)

## 3. Missing-value study

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing[missing["missing_count"] > 0])

missing["missing_percent"].sort_values().plot(
    kind="barh", figsize=(8, 5), title="Missing values by feature (%)"
)
plt.xlabel("Missing values (%)")
plt.show()

## 4. Target distribution and descriptive statistics

In [ ]:
target_counts = df["survived"].value_counts().sort_index()
target_percent = df["survived"].value_counts(normalize=True).sort_index().mul(100)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percent.round(2)
})
display(target_summary)

sns.countplot(data=df, x="survived")
plt.title("Survival target distribution")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.show()

numeric_cols = ["age", "fare", "sibsp", "parch"]
display(df[numeric_cols].describe().T)

## 5. Explore relationships with the target

In [ ]:
survival_by_sex = df.groupby("sex", observed=True)["survived"].agg(
    count="count", survival_rate="mean"
)
survival_by_class = df.groupby("pclass")["survived"].agg(
    count="count", survival_rate="mean"
)

display(survival_by_sex)
display(survival_by_class)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x="sex", y="survived", ax=axes[0], errorbar=None)
axes[0].set_title("Survival rate by sex")
sns.barplot(data=df, x="pclass", y="survived", ax=axes[1], errorbar=None)
axes[1].set_title("Survival rate by passenger class")
plt.tight_layout()
plt.show()

## 6. Investigate the suspicious feature: `alive`

First, inspect the cross-tabulation. If `alive` is simply another representation of the outcome, it should map directly to `survived`.

In [ ]:
display(pd.crosstab(df["alive"], df["survived"], margins=True))

mapping_check = (df["alive"].astype(str).str.lower() ==
                 df["survived"].map({0: "no", 1: "yes"}))
print("Rows where alive agrees with survived:", mapping_check.mean())
print("Unique alive values:", df["alive"].unique())
print("Unique survived values:", df["survived"].unique())

### Interpretation prompt

- Does `alive` contain information that would be known before the passenger's survival outcome?
- Is the relationship merely strong, or is it effectively a direct encoding of the target?
- Would a model using this feature be useful at the time the prediction is supposed to happen?

A feature can be highly predictive and still be invalid.

## 7. Model experiment: with and without target leakage

We compare a model that incorrectly includes `alive` with a model that excludes it. We keep the same train/test split and preprocessing approach for a fair demonstration.

In [ ]:
X_leaky = df[["pclass", "sex", "age", "fare", "sibsp", "parch", "embarked", "alive"]]
X_valid = df[["pclass", "sex", "age", "fare", "sibsp", "parch", "embarked"]]
y = df["survived"]

X_train_l, X_test_l, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42, stratify=y
)

# Use the exact same row split for the valid-feature experiment.
X_train_v = X_train_l.drop(columns="alive")
X_test_v = X_test_l.drop(columns="alive")

numeric_features = ["pclass", "age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "embarked"]

def build_pipeline():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000))
    ])

# Leaky model: alive is encoded as a categorical predictor.
leaky_cat = categorical_features + ["alive"]
def build_leaky_pipeline():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    preprocess = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, leaky_cat)
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000))
    ])

valid_model = build_pipeline()
leaky_model = build_leaky_pipeline()

valid_model.fit(X_train_v, y_train)
leaky_model.fit(X_train_l, y_train)

valid_pred = valid_model.predict(X_test_v)
leaky_pred = leaky_model.predict(X_test_l)

results = pd.DataFrame({
    "experiment": ["Valid features only", "Includes leaked feature: alive"],
    "test_accuracy": [
        accuracy_score(y_test, valid_pred),
        accuracy_score(y_test, leaky_pred)
    ]
})
display(results)

### Inspect the results

In [ ]:
print("Valid model")
print(classification_report(y_test, valid_pred))

print("Leaky model")
print(classification_report(y_test, leaky_pred))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, title in [
    (axes[0], valid_pred, "Valid features"),
    (axes[1], leaky_pred, "Leaked feature included")
]:
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d",
                cmap="Blues", cbar=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 8. Preprocessing leakage: fitting a scaler before splitting

A scaler learns statistical quantities such as the mean and standard deviation. If it is fitted on the complete dataset before the split, the test-set feature distribution contributes to those quantities.

We will compare the mean learned from the full dataset with the mean learned from training data only. The full-data mean is not necessarily dramatically different; the problem is that test information has influenced the learned transformation.

In [ ]:
X_num = df[["age", "fare", "sibsp", "parch"]].copy()
X_train_num, X_test_num = train_test_split(
    X_num, test_size=0.2, random_state=42
)

scaler_full = StandardScaler().fit(X_num)
scaler_train = StandardScaler().fit(X_train_num)

comparison = pd.DataFrame({
    "full_dataset_mean": scaler_full.mean_,
    "training_only_mean": scaler_train.mean_,
    "difference": scaler_full.mean_ - scaler_train.mean_
}, index=X_num.columns)

display(comparison)

### Correct versus incorrect pattern

In [ ]:
# WRONG: learns scaling parameters using both train and test rows.
wrong_scaler = StandardScaler()
X_all_scaled_wrong = wrong_scaler.fit_transform(X_num)
X_train_wrong, X_test_wrong, y_train_wrong, y_test_wrong = train_test_split(
    X_all_scaled_wrong, y, test_size=0.2, random_state=42, stratify=y
)

# RIGHT: learn parameters from training rows, then apply to test rows.
X_train_raw, X_test_raw, y_train_right, y_test_right = train_test_split(
    X_num, y, test_size=0.2, random_state=42, stratify=y
)
right_scaler = StandardScaler()
X_train_right = right_scaler.fit_transform(X_train_raw)
X_test_right = right_scaler.transform(X_test_raw)

print("Wrong scaler mean:", wrong_scaler.mean_)
print("Right scaler mean:", right_scaler.mean_)

**Note:** On this small example, the difference in model score from global scaling may be small or even absent. That does not make the workflow valid. Leakage is a methodological flaw even when it does not visibly change one particular score. In real workflows, leakage can distort validation, feature selection, imputation, scaling, dimensionality reduction and hyperparameter tuning.

## 9. The safe workflow: use a Pipeline

In [ ]:
# The pipeline is fitted only on training rows.
safe_model = build_pipeline()
safe_model.fit(X_train_v, y_train)
safe_predictions = safe_model.predict(X_test_v)

print("Pipeline test accuracy:", accuracy_score(y_test, safe_predictions))

## 10. Findings and interpretation

Complete these after running the notebook:

1. What proportion of passengers survived?
2. Which features have missing values, and how much is missing?
3. How do survival rates differ by sex and passenger class?
4. What does the `alive` vs `survived` cross-tabulation show?
5. How much did test accuracy change when `alive` was included?
6. How did the full-data scaler mean differ from the training-only mean?
7. Why is the pipeline safer?

### Three questions every data scientist should ask

- Would this feature be available at the exact moment the prediction is made?
- Was any information from validation/test data used to learn preprocessing or select features?
- Does the validation strategy match how the model will be used in the real world?

## 11. Key takeaways

- Data leakage makes evaluation overly optimistic and can cause poor real-world performance.
- Target leakage happens when a predictor contains information about the outcome or information only available after the outcome.
- Preprocessing must be learned from training data only.
- Use `fit_transform` on training data and `transform` on validation/test data.
- Pipelines reduce the chance of accidental leakage.
- A high score is not proof of a good model; investigate how that score was obtained.

## References

- [Scikit-learn: Common pitfalls and data leakage](https://scikit-learn.org/stable/common_pitfalls.html)
- [Scikit-learn: Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Seaborn Titanic dataset](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv)

**Day 02 principle:** A model should learn from information that would genuinely be available when making its prediction — nothing more.